# BPL Windows and Neighbours

In [1]:
!pip install -U basedpl

In [2]:
%load_ext basedpl.notebooks

In [3]:
import basedpl
basedpl.__version__

'0.1.91'

## Looking at each item together with its neighbours

Many calculations look at an item **together with the items next to it**: the change from one day to the next, a 3-day moving average, or the 8 cells around a square on a grid. This dialog starts with pairs and sliding *windows* on a list, then moves to *neighbourhoods* with stencil, and ends with Conway's Game of Life.

BPL docs: [Windows `↕`](https://answerdotai.github.io/basedpl/glyphs/up-down-arrow.html) · [Stencil `⌺`](https://answerdotai.github.io/basedpl/glyphs/quad-diamond.html)

Glyphs and ideas this dialog teaches:

| Glyph | Plain English |
|---|---|
| `↕v` | pairs: each item with the next one, one pair per row (typed **Alt-K**) |
| `3↕v` | windows: every run of 3 items in a row, one window per row |
| `f⌺3` | stencil: apply `f` to each item's neighbourhood of 3, centred on it. The answer has one result per item (typed **Alt-q**, then **Backtick**) |
| `["size":… "edge":"clamp"]` | options for windows: what to use past the ends (`"clamp"` repeats the end item, `"wrap"` goes round to the other end) |
| `f⌺s⁝g` | stencil on a table: `s←3 3` is a 3×3 neighbourhood |

Other glyphs used along the way:

| Glyph | Plain English |
|---|---|
| `←` | store a value in a name |
| `⍝` | comment: BPL ignores the rest of the line |
| `+/` `⌈/` `⌊/` | total, biggest, smallest (of each row of a table) |
| `-⍨/` | subtract, with the sides swapped: second minus first |
| `÷` `≢` | divide, count |
| `,` | flatten a table into one list |
| `ρ` | reshape: `4 4ρ↦16` is the numbers 1 to 16 in a 4×4 table |
| `[… ◇ …]` | a table written row by row |
| `= ∧ ∨` | equal, and, or (answers `$t` or `$f`) |
| `".#".[g]` | read characters at positions: 0 gives `.`, 1 gives `#` |
| `⁝` | keeps two things apart, so `3` stays the size and `temps` the data |
| `⍣` | repeat a function |
| `¨` | each |

### The data

A week of midday temperatures (°C), Monday to Sunday.

In [4]:
%%bpl
temps←12 15 11 18 20 14 9      ⍝ Mon to Sun

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

### Pairs and windows

#### 1. Pairs: `↕`

- `↕temps` puts each day next to the following day: one pair per row, so 7 days give 6 pairs.
- `-⍨/` on each row gives "second minus first": the change from one day to the next.

In [5]:
%%bpl
temps
↕temps            ⍝ Mon-Tue, Tue-Wed, … one pair per row
-⍨/↕temps         ⍝ day-to-day change: 3 ¯4 7 2 ¯6 ¯5

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

12 15 11 18 20 14 9

┌→────┐
↓12 15│
│15 11│
│11 18│
│18 20│
│20 14│
│14  9│
└~────┘

3 ¯4 7 2 ¯6 ¯5

Functions act on data. Operators take a function and make a new function from it.

| Glyph | Kind | Name | What it does here |
|---|---|---|---|
| `↕` | function | **pairs** (with one argument), **windows** (with two) | `↕temps` pairs each item with the next, so 7 temperatures give 6 rows |
| `-` | function | **subtract** (with two arguments), **negate** (with one) | the arithmetic step inside each pair |
| `⍨` | monadic operator | **commute** (with two arguments), **self** (with one) | applies to the function on its left: `-⍨` is subtract with the sides swapped, so `a -⍨ b` is `b - a` |
| `/` | monadic operator | **reduce** | applies to the function on its left: `f/` puts `f` between the items of each row. On a pair `a b` this gives `a f b` |
| `⍝` | syntax (neither) | **comment** | BPL ignores the rest of the line |

Reading `-⍨/↕temps` right to left:
1. `↕temps`: the 6 rows of pairs.
2. `-⍨`: the operator `⍨` turns `-` into "swapped subtract".
3. `(-⍨)/`: the operator `/` turns that into "reduce each row with swapped subtract", so `12 15` becomes `12 -⍨ 15`, which is `15 - 12`, or `3`.

Result: the day-to-day changes `3 ¯4 7 2 ¯6 ¯5`.

Operators bind to the function on their left before any function is applied, so `-⍨/` is one derived function.

#### 2. Windows of 3

- `3↕temps` gives every run of 3 days in a row, one *window* per row. Each window slides along by one day.
- 7 days give 5 windows: Mon-Wed, Tue-Thu, and so on to Fri-Sun.

In [6]:
%%bpl
temps
3↕temps           ⍝ every 3 days in a row

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

12 15 11 18 20 14 9

┌→───────┐
↓12 15 11│
│15 11 18│
│11 18 20│
│18 20 14│
│20 14  9│
└~───────┘

#### 3. A moving average

- `+/` on a table adds up each **row**, so `+/3↕temps` is the total of each 3-day window.
- Divide by 3 for a 3-day moving average, which smooths out the ups and downs.
- `⌈/3↕temps` is the hottest day in each window.

In [7]:
%%bpl
3↕temps
+/3↕temps         ⍝ 3-day totals
(+/3↕temps)÷3     ⍝ 3-day moving average
⌈/3↕temps         ⍝ hottest in each window

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→───────┐
↓12 15 11│
│15 11 18│
│11 18 20│
│18 20 14│
│20 14  9│
└~───────┘

38 44 49 52 43

12.66666667 14.66666667 16.33333333 17.33333333 14.33333333

15 18 20 20 20

### Neighbours: stencil

#### 4. One answer per day: `⌺`

- Windows give **fewer** answers than days (5, not 7). Often you want one answer per day, worked out from each day and its neighbours.
- `+/⌺3` (stencil) adds up each day's neighbourhood of 3: the day before, the day itself and the day after.
- `⁝` keeps the size `3` apart from the data `temps`.
- At the ends there's no day before Monday or after Sunday, so stencil uses 0 there. That makes the first and last totals too small.

In [8]:
%%bpl
temps
+/⌺3⁝temps        ⍝ one total per day: the ends include a 0

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

12 15 11 18 20 14 9

27 38 44 49 52 43 23

#### 5. What happens at the edges

- An options record chooses what stencil uses past the ends. `"edge":"clamp"` repeats the end day: Monday's neighbourhood becomes Mon, Mon, Tue.
- Compare the moving averages: with the default 0 padding, Monday's is far too low (9). With clamp it's sensible (13).

In [9]:
%%bpl
mean←{(+/ω)÷≢ω}
mean⌺3⁝temps                          ⍝ padding with 0: low at the ends
clamp←["size":3 "edge":"clamp"]
mean⌺clamp⁝temps                      ⍝ repeat the end day instead

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

9 12.66666667 14.66666667 16.33333333 17.33333333 14.33333333 7.666666667

13 12.66666667 14.66666667 16.33333333 17.33333333 14.33333333 10.66666667

### Grids

#### 6. Neighbourhoods on a table

- Stencil works on tables too. `s←3 3` asks for a 3×3 neighbourhood: each cell with the 8 cells around it.
- Inside the braces, `ω` is a 3×3 block. `,` flattens it into a list, and `+/` adds it up.
- The size goes in a name (`s`) because `3 3` written straight inside an options record would split into two items.

In [10]:
%%bpl
m←4 4ρ↦16        ⍝ the numbers 1 to 16
m
s←3 3            ⍝ a 3×3 neighbourhood
{+/,ω}⌺s⁝m       ⍝ each cell plus its 8 neighbours

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→──────────┐
↓ 1  2  3  4│
│ 5  6  7  8│
│ 9 10 11 12│
│13 14 15 16│
└~──────────┘

┌→──────────┐
↓14 24 30 22│
│33 54 63 45│
│57 90 99 69│
│46 72 78 54│
└~──────────┘

#### 7. Counting live neighbours

- A grid of 1s and 0s can stand for live and dead cells.
- The 3×3 total includes the cell itself, so subtract the grid to count just the **neighbours**.
- `show` draws the grid with `#` for a live cell and `.` for a dead one.

In [11]:
%%bpl
g←[0 0 0 0 0◇0 0 1 0 0◇0 0 1 0 0◇0 0 1 0 0◇0 0 0 0 0]   ⍝ a short vertical line
g
s
show←{".#".[ω]}          ⍝ 0 → .  and  1 → #
show g
n←({+/,ω}⌺s⁝g)-g         ⍝ live neighbours of each cell
n

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→────────┐
↓0 0 0 0 0│
│0 0 1 0 0│
│0 0 1 0 0│
│0 0 1 0 0│
│0 0 0 0 0│
└~────────┘

3 3

┌→────┐
↓.....│
│..#..│
│..#..│
│..#..│
│.....│
└─────┘

┌→────────┐
↓0 1 1 1 0│
│0 2 1 2 0│
│0 3 2 3 0│
│0 2 1 2 0│
│0 1 1 1 0│
└~────────┘

### The Game of Life

#### 8. One step of Life

Conway's Game of Life has two rules, applied to every cell at once:

- A **live** cell with 2 or 3 live neighbours stays alive. Otherwise it dies.
- A **dead** cell with exactly 3 live neighbours comes alive.

In BPL: alive next step if `n=3`, **or** (`∨`) alive now **and** (`∧`) `n=2`. The vertical line flips to a horizontal one, then back again. It's called a *blinker*.

In [12]:
%%bpl
life←{n←({+/,ω}⌺s⁝ω)-ω ◇ (n=3)∨ω∧n=2}
show life g          ⍝ one step: horizontal
show life life g     ⍝ two steps: vertical again

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→────┐
↓.....│
│.....│
│.###.│
│.....│
│.....│
└─────┘

┌→────┐
↓.....│
│..#..│
│..#..│
│..#..│
│.....│
└─────┘

#### 9. A glider on a wrap-around grid

- A *glider* is a shape that travels across the grid: every 4 steps it ends up 1 cell down and 1 cell right.
- With `"edge":"wrap"`, the grid wraps around: off the right edge means back on the left. So the glider never hits a wall.
- `lifew⍣4⁝glider` runs 4 steps, and `⍣(ι3)` keeps the first 3 generations so you can watch it move.
- On this 6×6 grid, after 24 steps it's back exactly where it started.

In [13]:
%%bpl
wrap←["size":s "edge":"wrap"]
lifew←{n←({+/,ω}⌺wrap⁝ω)-ω ◇ (n=3)∨ω∧n=2}
glider←[0 1 0 0 0 0◇0 0 1 0 0 0◇1 1 1 0 0 0◇0 0 0 0 0 0◇0 0 0 0 0 0◇0 0 0 0 0 0]
show glider
show¨lifew⍣(ι3)⁝glider     ⍝ generations 0, 1, 2
show lifew⍣4⁝glider        ⍝ 4 steps: moved one cell down and right
(lifew⍣24⁝glider)≡glider       ⍝ $t: back where it started

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→─────┐
↓.#....│
│..#...│
│###...│
│......│
│......│
│......│
└──────┘

┌→───────────────────────────┐
│ ┌→─────┐ ┌→─────┐ ┌→─────┐ │
│ ↓.#....│ ↓......│ ↓......│ │
│ │..#...│ │#.#...│ │..#...│ │
│ │###...│ │.##...│ │#.#...│ │
│ │......│ │.#....│ │.##...│ │
│ │......│ │......│ │......│ │
│ │......│ │......│ │......│ │
│ └──────┘ └──────┘ └──────┘ │
└∊───────────────────────────┘

┌→─────┐
↓......│
│..#...│
│...#..│
│.###..│
│......│
│......│
└──────┘

$t

**How the glider "moves"**

Nothing in the grid actually moves. Each step, some `#`s die and new ones are born, following the two Life rules. It's like a stadium wave: no one changes seat, but the wave still travels.

```
gen 0    gen 1    gen 2    gen 3    gen 4
.#....   ......   ......   ......   ......
..#...   #.#...   ..#...   .#....   ..#...
###...   .##...   #.#...   ..##..   ...#..
......   .#....   .##...   .##...   .###..
```

- The shape goes through 4 different forms.
- At **gen 4** you get the gen 0 shape again, but **1 down and 1 right**.
- Each cycle of 4 steps repeats that shift.

**Why it shifts:** the shape is lopsided. On its bottom-right side there are spots with exactly 3 live neighbours, so new cells are born there. On the top-left side, cells have too few neighbours, so they die. It grows on one side and shrinks on the other, so it travels toward the growing side.

### Your turn

1. Find the biggest rise from one day to the next.
   *Hint:* `⌈/` of step 1's changes.
2. Work out the 2-day average for each pair of days.
   *Hint:* `2↕`, then `+/` and `÷2`.
3. Find the coldest day in each 3-day window.
   *Hint:* `⌊/`.
4. Use stencil with `"clamp"` to find each day's highest temperature among itself and its neighbours.
   *Hint:* `⌈/⌺clamp⁝temps`.
5. A 2×2 square of live cells is a *block*. Make one in the middle of a 4×4 grid and check that `life` leaves it unchanged.